## XBG Replication - Fraud Detection

In [ ]:
#General imports
import pandas as pd
from datetime import timedelta
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#DB2-specific imports
import pyodbc
from getpass import getuser, getpass

#ML-specific imports
from sklearn.model_selection import TimeSeriesSplit #ask t if they use cross-validation or not, probably very specific
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, TargetEncoder
from scipy.sparse import hstack
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay #should be enough right? Or can we disregard the classification report? 

In [ ]:
print("lets get this bread")

## Importing the data files and necessary packages

Here the data is imported. Due to sensitivity issues, the improt cannot be shown

## Initial EDA

The motivation behind this initial EDA is to explore factors like:

* Data quality errors
* Missing values or faulty datatypes
* Data shape and basic statistics

Once this is done, data cleaning, feature engineering, post-cleaning EDA and encoding follow

In [ ]:
pd.set_option("display.max_columns", None) #allows us to see full feature-space of the dataframe

In [ ]:
df.head()

In [ ]:
assert (df["COUNTERENTITYID"] == df["COUNTERPARTYID"]).all()
assert (df["ACCOUNTID"] == df["ACCOUNTENTITYID"]).all() #confirms that the columns duplicate eachother and hence struggle to add any information


In [ ]:

single_value_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with only one unique value:")
print(single_value_cols) #these columns are zero variance and provide no info. However, CONFIRMEDRISK is kept since that is our target variable

In [ ]:
#Checking transaction-distribution across dates, we want it approximetly equal
 
monthly_counts = df.groupby(df['EVENTTIME'].dt.to_period('M')).size() 
print("Counts per month:") 
print(monthly_counts)

In [ ]:
monthly_counts = df.loc[df["CONFIRMEDRISK"] == True].groupby(df['EVENTTIME'].dt.to_period('M')).size()/df.groupby(df['EVENTTIME'].dt.to_period('M')).size() 
print("Counts per month:") 
print(monthly_counts)

In [ ]:
monthly_counts = df.loc[df["CONFIRMEDRISK"] == True].groupby(df['EVENTTIME'].dt.to_period('M')).size()
print(monthly_counts)


In [ ]:
#Investigating account-related columns, will form the basis of encoding for later
account_columns = df.columns[:6]  #First 5 columns

for col in account_columns:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
df["ACCOUNTBRANCHID"].value_counts().head(20).values.sum() / len(df) #briliant, we can do top 20 and still retain a bunch of data

In [ ]:
# Get multiple column names
financial_cols = df.columns[6:11]  #First 5 columns

for col in financial_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
specific_columns =  [
    'CUSTOMERTYPE', 'DESTINATIONCOUNTRY', 'IPADDRESS', 'USERAGENTSTRING', 
    'DEVICEENTITYID', 'DEVICEID', 'EVENTTIME', 'MSGSTATUS', 'PAYMENTCLEARING', 
    'PAYMENTMETHOD', 'PAYMENTSUBMETHOD', 'TRANSACTIONID', 'TRANSACTIONONUS', 
    'EXCEPTIONRULE', 'INTERNATIONALFLAG'
]

for col in specific_columns:
    print(f"\n{col}:")
    print(df[col].value_counts())

## GENERAL DATA CLEANING AND PREPARATION

Here, we start cleaning and preparing the data. This is a general procedure which will be followed by Danske Bank specific guidelines.

In [ ]:
#Dropping redundant columns
df.drop(columns=["ACCOUNTAGENTID", 
                 "EXCEPTIONRULE", 
                 "CUSTOMERTYPE", 
                 "MSGSTATUS",
                 "PAYMENTMETHOD", 
                 "IPADDRESS", 
                 "USERAGENTSTRING", 
                 "DEVICEID", 
                 "DEVICEENTITYID", 
                 "CUSTOMERENTITYID", 
                 "COUNTERENTITYID", 
                 "ACCOUNTENTITYID", 
                 "ACCOUNTIDFORMAT", 
                 "BASECURRENCY", 
                 "VALUE"], inplace = True) #might keep transactionID for Carls sake if he needs it in GNN

* Columns CONFIRMEDRISK, INTERNATIONALFLAG and TRANSACTIONOUS are transformed into booleans.

In [ ]:
df["INTERNATIONALFLAG"].map({"true": True}).value_counts() #seems to be a problem with trailing whitespace

In [ ]:
df["INTERNATIONALFLAG"] = df["INTERNATIONALFLAG"].str.strip().str.lower().map({"true": True, "false": False}) #using accessor function so that the whitespace can be trimmed away

In [ ]:
df["TRANSACTIONONUS"] = df["TRANSACTIONONUS"].map({"true": True, "false": False})

Verify effect

In [ ]:
df.info()

In [ ]:
df.select_dtypes("object")

The remaining object-dtypes will be handled via encoding. 

Since the datatypes are correct and no apperent data-quality issues persist, we continue with post cleaning eda. After that follows feature-engineering and encoding, before finally fitting the model

In [ ]:
df.info()

## Post cleaning eda

The motivation behind the post cleaning EDA is to confirm whether the cleaning process has been sufficient and to reveal insights for feature-engineering. The post cleaning EDA will explore factors such as:

* General dataset charachteristics
* Distribution checks
* Fraud Pattern Analysis
* Temporal and behavioral patterns

It is important to bear in mind that these insights capture patterns which occur in our sample

In [ ]:
#investigating class-imbalance

len(df.loc[df["CONFIRMEDRISK"] == True]) / len(df) 

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format) #making sure that summary statistics are displayed in a readable manner

In [ ]:
df.loc[df["CONFIRMEDRISK"] == False]["BASEVALUE"].describe()

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True]["BASEVALUE"].describe()

In [ ]:
df["BASEVALUE"].describe()

In [ ]:
#Visualizing the distribution of values for thesis purpose

y = sns.displot(data = df, x = "BASEVALUE", hue = "CONFIRMEDRISK", kind = "kde") #clear outliers, need to make one with buckets
y.fig.suptitle("Transactions Distribution", y = 1.03)

In [ ]:
#Create separate boxplots
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

#Non-fraud boxplot
df[df["CONFIRMEDRISK"] == False]["BASEVALUE"].plot(kind='box', ax=axes[0], title='Non-Fraudulent Transactions')

#Fraud boxplot  
df[df["CONFIRMEDRISK"] == True]["BASEVALUE"].plot(kind='box', ax=axes[1], title='Fraudulent Transactions')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(
    np.log1p(df[df["CONFIRMEDRISK"] == False]["BASEVALUE"]),
    bins=50,
    alpha=0.7,
    label="Non-Fraud",
    density=True,          
    color="steelblue",
)

ax.hist(
    np.log1p(df[df["CONFIRMEDRISK"] == True]["BASEVALUE"]),
    bins=50,
    alpha=0.7,
    label="Fraud",
    density=True,          
    color="seagreen",
)

ax.set_xlabel("Log(BASEVALUE + 1)")
ax.set_ylabel("Density")
ax.set_title("Class-conditional distribution of log-transformed transaction value")
ax.legend()
plt.tight_layout()
plt.show()



In [ ]:

non_fraud_99th = df[df["CONFIRMEDRISK"] == False]["BASEVALUE"].quantile(0.99)
fraud_99th = df[df["CONFIRMEDRISK"] == True]["BASEVALUE"].quantile(0.99)

print(f"Non-fraud 99th percentile: €{non_fraud_99th:.2f}")
print(f"Fraud 99th percentile: €{fraud_99th:.2f}") 

In [ ]:
non_fraud_99th = df[df["CONFIRMEDRISK"] == False]["BASEVALUE"].quantile(0.95)
fraud_99th = df[df["CONFIRMEDRISK"] == True]["BASEVALUE"].quantile(0.95)

print(f"Non-fraud 95th percentile: €{non_fraud_99th:.2f}")
print(f"Fraud 95th percentile: €{fraud_99th:.2f}") 

In [ ]:
non_fraud_99th = df[df["CONFIRMEDRISK"] == False]["BASEVALUE"].quantile(0.90)
fraud_99th = df[df["CONFIRMEDRISK"] == True]["BASEVALUE"].quantile(0.90)

print(f"Non-fraud 90th percentile: €{non_fraud_99th:.2f}")
print(f"Fraud 90th percentile: €{fraud_99th:.2f}") 

In [ ]:
non_fraud_99th = df[df["CONFIRMEDRISK"] == False]["BASEVALUE"].quantile(0.90)
fraud_99th = df[df["CONFIRMEDRISK"] == True]["BASEVALUE"].quantile(0.55)

print(f"Non-fraud 90th percentile: €{non_fraud_99th:.2f}")
print(f"Fraud 90th percentile: €{fraud_99th:.2f}") 

In [ ]:
tx_per_account = df["ACCOUNTID"].value_counts()

print(tx_per_account.describe(percentiles=[.5, .75, .9, .95, .99]))

print(f"Accounts with exactly 1 transaction: {(tx_per_account == 1).sum()}")
print(f"Accounts with more than 5 transactions: {(tx_per_account > 5).sum()}")
print(f"Accounts with more than 10 transactions: {(tx_per_account > 10).sum()}")
print(f"Accounts with 5 or fewer transactions: {(tx_per_account <= 5).sum()}")

## Internationally focused EDA

We probably want to see where the two classes differ to discover signals

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending = False)

In [ ]:
top_5 = df.loc[df["CONFIRMEDRISK"] == True].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending = False).index[0:100]

In [ ]:
top_5

In [ ]:
df.loc[(df["CONFIRMEDRISK"] == True) & (df["DESTINATIONCOUNTRY"].isin(top_5))]["DESTINATIONCOUNTRY"].value_counts() 

In [ ]:
df.loc[df["CONFIRMEDRISK"] == False].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending = False)

In [ ]:
top_5 = df.loc[df["CONFIRMEDRISK"] == False].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending = False).index[0:5] #getting top 5, seeing their occurence
df.loc[(df["CONFIRMEDRISK"] == False) & (df["DESTINATIONCOUNTRY"].isin(top_5))]["DESTINATIONCOUNTRY"].value_counts() 

In [ ]:
fraud_international = df.loc[df["CONFIRMEDRISK"] == True]["INTERNATIONALFLAG"].value_counts()

import matplotlib.pyplot as plt

plt.bar(fraud_international.index, fraud_international.values)
plt.title("International and Domestic payments - Fraudulent transactions", y = 1.03)
plt.xticks(fraud_international.index, ["False", "True"])
plt.show() 

In [ ]:
fraud_international = df.loc[df["CONFIRMEDRISK"] == False]["INTERNATIONALFLAG"].value_counts()


#looking at international payments
import matplotlib.pyplot as plt

plt.bar(fraud_international.index, fraud_international.values)
plt.title("International and Domestic payments - Regular transactions", y = 1.03)
plt.xticks(fraud_international.index, ["False", "True"])
plt.show()

In [ ]:


regular = df.loc[df["CONFIRMEDRISK"] == False, "INTERNATIONALFLAG"].value_counts()
fraud   = df.loc[df["CONFIRMEDRISK"] == True,  "INTERNATIONALFLAG"].value_counts()


regular = regular.reindex([False, True], fill_value=0)
fraud   = fraud.reindex([False, True], fill_value=0)


regular_prop = regular / regular.sum()
fraud_prop   = fraud   / fraud.sum()


categories = ["Domestic", "International"]
x = np.arange(len(categories))
width = 0.38

fig, ax = plt.subplots(figsize=(7, 5))
bars_reg   = ax.bar(x - width/2, regular_prop.values, width,
                    label=f"Legitimate (n = {regular.sum():,})",
                    color="#blue", edgecolor="black", linewidth=0.5)
bars_fraud = ax.bar(x + width/2, fraud_prop.values, width,
                    label=f"Fraudulent (n = {fraud.sum():,})",
                    color="#red", edgecolor="black", linewidth=0.5)


def annotate(bars, counts):
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.1%}\n(n = {count:,})",
                ha="center", va="bottom", fontsize=9)

annotate(bars_reg,   regular.values)
annotate(bars_fraud, fraud.values)

ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel("Within-class proportion")
ax.set_ylim(0, 1.1)
ax.set_title("Distribution of International Payments by Transaction Class",
             pad=12)
ax.legend(frameon=False, loc="upper right")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.show()


In [ ]:
regular = df.loc[df["CONFIRMEDRISK"] == False, "INTERNATIONALFLAG"].value_counts()
fraud   = df.loc[df["CONFIRMEDRISK"] == True,  "INTERNATIONALFLAG"].value_counts()

regular = regular.reindex([False, True], fill_value=0)
fraud   = fraud.reindex([False, True], fill_value=0)

regular_prop = regular / regular.sum()
fraud_prop   = fraud   / fraud.sum()


classes = [f"Legitimate\n(n = {regular.sum():,})",
           f"Fraudulent\n(n = {fraud.sum():,})"]
x = np.arange(len(classes))
width = 0.38

fig, ax = plt.subplots(figsize=(7, 5))
bars_dom = ax.bar(x - width/2,
                  [regular_prop[False], fraud_prop[False]],
                  width, label="Domestic",
                  color="blue", edgecolor="black", linewidth=0.5)
bars_int = ax.bar(x + width/2,
                  [regular_prop[True], fraud_prop[True]],
                  width, label="International",
                  color="blue", edgecolor="black", linewidth=0.5)

# Annotate bars with percentage and absolute count
def annotate(bars, counts):
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f"{height:.1%}\n(n = {count:,})",
                ha="center", va="bottom", fontsize=9)

annotate(bars_dom, [regular[False], fraud[False]])
annotate(bars_int, [regular[True],  fraud[True]])

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("Within-class proportion")
ax.set_ylim(0, 1.15)
ax.set_title("Distribution of International Payments by Transaction Class",
             pad=12)
ax.legend(frameon=False, loc="upper right", title="Payment type")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.show()


contingency = pd.crosstab(df["INTERNATIONALFLAG"], df["CONFIRMEDRISK"])
contingency.index = ["Domestic", "International"]
contingency.columns = ["Legitimate", "Fraudulent"]
contingency["Fraud rate"] = (
    contingency["Fraudulent"] / contingency.sum(axis=1)
).map(lambda p: f"{p:.3%}")

print("\nConditional fraud rates:")
print(contingency)


In [ ]:
df.loc[df["INTERNATIONALFLAG"] == True]["CONFIRMEDRISK"].value_counts() #fucking great

In [ ]:
country_stats = (df.groupby("DESTINATIONCOUNTRY").agg(
    n_transactions = ('CONFIRMEDRISK', 'size'),
    n_fraud = ('CONFIRMEDRISK', 'sum')
).assign(fraud_rate = lambda d: d['n_fraud'] / d['n_transactions']).sort_values('n_transactions', ascending = False)
)

top20 = country_stats.head(20)
print(top20)

In [ ]:
gr_subset = df.loc[df["DESTINATIONCOUNTRY"] == 'GR']
print(gr_subset["CONFIRMEDRISK"].value_counts())
print(gr_subset["EVENTTIME"].describe())
print(gr_subset.groupby("COUNTERPARTYID")["CONFIRMEDRISK"].agg(["mean","size"]).sort_values("size", ascending = False).head(10))

In [ ]:
gr_subset = df.loc[df["DESTINATIONCOUNTRY"] == 'MT']
print(gr_subset["CONFIRMEDRISK"].value_counts())
print(gr_subset["EVENTTIME"].describe())
print(gr_subset.groupby("COUNTERPARTYID")["CONFIRMEDRISK"].agg(["mean","size"]).sort_values("size", ascending = False).head(10))

In [ ]:
gr_subset = df.loc[df["DESTINATIONCOUNTRY"] == 'PL']
print(gr_subset["CONFIRMEDRISK"].value_counts())
print(gr_subset["EVENTTIME"].describe())
print(gr_subset.groupby("COUNTERPARTYID")["CONFIRMEDRISK"].agg(["mean","size"]).sort_values("size", ascending = False).head(10))

In [ ]:
fraud_international = df.loc[df["CONFIRMEDRISK"] == True]["INTERNATIONALFLAG"].value_counts()

In [ ]:
df.loc[df["INTERNATIONALFLAG"] == True]["DESTINATIONCOUNTRY"].value_counts()

In [ ]:
#is there a discrepancy between the legit and fraudulent ones?

df.loc[(df["INTERNATIONALFLAG"] == True) & (df["CONFIRMEDRISK"] == True)]["DESTINATIONCOUNTRY"].value_counts()[0:10]

In [ ]:
df.loc[(df["INTERNATIONALFLAG"] == True) & (df["CONFIRMEDRISK"] == True)][["DESTINATIONCOUNTRY", "BASEVALUE"]]

In [ ]:
mask = (df["INTERNATIONALFLAG"]) & (df["CONFIRMEDRISK"]) 
medians = df.loc[mask].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median()
medians.sort_values(ascending = False)

In [ ]:
df.loc[(df["INTERNATIONALFLAG"] == True) & (df["CONFIRMEDRISK"] == False)]["DESTINATIONCOUNTRY"].value_counts()[0:10]

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True]["ACCAGENTCOUNTRY"].value_counts()

In [ ]:
df["ACCAGENTCOUNTRY"].value_counts()

In [ ]:
df["TRANSACTIONONUS"].value_counts()

In [ ]:
fraud_on_us = df.loc[df["CONFIRMEDRISK"] == True]["TRANSACTIONONUS"].value_counts()

plt.bar(fraud_on_us.index, fraud_on_us.values)
plt.title("Intrabank transactions - Fraud", y = 1.03)
plt.xticks(fraud_on_us.index, ["False", "True"])
plt.show()

In [ ]:
fraud_on_us = df.loc[df["CONFIRMEDRISK"] == False]["TRANSACTIONONUS"].value_counts()

plt.bar(fraud_on_us.index, fraud_on_us.values)
plt.title("Intrabank transactions - Regular Transactions", y = 1.03)
plt.xticks(fraud_on_us.index, ["False", "True"])
plt.show()

In [ ]:
df.info()

In [ ]:

df[["DESTINATIONCOUNTRY", "BASEVALUE"]].sort_values(by = "BASEVALUE", ascending = False)

In [ ]:
df.groupby("DESTINATIONCOUNTRY")["BASEVALUE"].mean().sort_values(ascending = False) 

In [ ]:

top_destination = df.groupby("DESTINATIONCOUNTRY")["BASEVALUE"].mean().sort_values(ascending=False).head()


plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Average Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 Destination Countries by Average Transaction Value')

In [ ]:
tx_per_account = df["ACCOUNTID"].value_counts()

print(tx_per_account.describe(percentiles=[.5, .75, .9, .95, .99]))

print(f"Accounts with exactly 1 transaction: {(tx_per_account == 1).sum()}")
print(f"Accounts with more than 5 transactions: {(tx_per_account > 5).sum()}")
print(f"Accounts with more than 10 transactions: {(tx_per_account > 10).sum()}")
print(f"Accounts with 5 or fewer transactions: {(tx_per_account <= 5).sum()}")

In [ ]:
tx_count_per_row = df['ACCOUNTID'].map(tx_per_account)
print("Rows from accounts with 1 tx:", (tx_count_per_row == 1).mean())
print("Rows from accounts with ≤5 tx:", (tx_count_per_row <= 5).mean())
print("Rows from accounts with >10 tx:", (tx_count_per_row > 10).mean())

fraud_accounts = df.loc[df['CONFIRMEDRISK'] == True, 'ACCOUNTID']
fraud_tx_count = fraud_accounts.map(tx_per_account)
print(fraud_tx_count.describe(percentiles=[.5, .75, .9]))
print("Fraud cases from accounts with 1 tx:", (fraud_tx_count == 1).mean())
print("Fraud cases from accounts with ≤5 tx:", (fraud_tx_count <= 5).mean())
print("Fraud cases from accounts with >10 tx:", (fraud_tx_count > 10).mean()) #kinda confirms the case for velocity


In [ ]:
tx_count_per_row = df.loc[df["CONFIRMEDRISK"] == False]['ACCOUNTID'].map(tx_per_account)
print("Rows from accounts with 1 tx:", (tx_count_per_row == 1).mean())
print("Rows from accounts with ≤5 tx:", (tx_count_per_row <= 5).mean())
print("Rows from accounts with >10 tx:", (tx_count_per_row > 10).mean())

fraud_accounts = df.loc[df['CONFIRMEDRISK'] == True, 'ACCOUNTID']
fraud_tx_count = fraud_accounts.map(tx_per_account)
print(fraud_tx_count.describe(percentiles=[.5, .75, .9]))
print("Fraud cases from accounts with 1 tx:", (fraud_tx_count == 1).mean())
print("Fraud cases from accounts with ≤5 tx:", (fraud_tx_count <= 5).mean())
print("Fraud cases from accounts with >10 tx:", (fraud_tx_count > 10).mean()) #kinda confirms the case for velocity


In [ ]:
df.groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending = False) 

In [ ]:
top_destination = df.groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending=False).head()

plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Median Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 Destination Countries by Median Transaction Value')

In [ ]:
top_destination = df.loc[df["CONFIRMEDRISK"] == True].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].mean().sort_values(ascending=False).head()


plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Median Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 Destination Countries by Median Transaction Value')

In [ ]:

top_destination = df.loc[df["CONFIRMEDRISK"] == True].groupby("DESTINATIONCOUNTRY")["BASEVALUE"].median().sort_values(ascending=False).head()

plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Median Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 Destination Countries by Median Transaction Value')

In [ ]:
df.groupby("DESTINATIONCOUNTRY")["TRANSACTIONID"].count().sort_values(ascending = False).head() #Totally different picture

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True].groupby("DESTINATIONCOUNTRY")["TRANSACTIONID"].count().sort_values(ascending = False).head() #Kinda the same

In [ ]:
df.loc[df["CONFIRMEDRISK"] == False].groupby("DESTINATIONCOUNTRY")["TRANSACTIONID"].count().sort_values(ascending = False).head() #Kinda the same

In [ ]:


top_destination = df.loc[df["CONFIRMEDRISK"] == True].groupby("CURRENCY")["BASEVALUE"].mean().sort_values(ascending=False).head()


plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Median Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 currencies by Median Transaction Value')

In [ ]:


top_destination = df.loc[df["CONFIRMEDRISK"] == True].groupby("CURRENCY")["BASEVALUE"].median().sort_values(ascending=False).head()


plt.figure(figsize=(10, 6))
plt.bar(top_destination.index, top_destination.values)
plt.xlabel('Median Transaction Value (EUR)')
plt.ylabel('Destination Country')
plt.title('Top 5 Destination Countries by Median Transaction Value')

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True].groupby("CURRENCY")["TRANSACTIONID"].count().sort_values(ascending = False) 

In [ ]:
df.loc[df["CONFIRMEDRISK"] == False].groupby("CURRENCY")["TRANSACTIONID"].count().sort_values(ascending = False) 

In [ ]:
len(df.loc[(df["CONFIRMEDRISK"] == True) & (df["INTERNATIONALFLAG"] == True)]) / len(df.loc[df["CONFIRMEDRISK"] == True]) 

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True]["ACCAGENTCOUNTRY"].value_counts() 

In [ ]:
df.loc[df["CONFIRMEDRISK"] == True]["DESTINATIONCOUNTRY"].value_counts().head(10)

In [ ]:
len(df.loc[(df["INTERNATIONALFLAG"] == True) & (df["CONFIRMEDRISK"] == True)]) / len(df.loc[df["INTERNATIONALFLAG"] == True]) 

In [ ]:
len(df.loc[(df["INTERNATIONALFLAG"] == False) & (df["CONFIRMEDRISK"] == True)]) / len(df.loc[df["INTERNATIONALFLAG"] == False]) 

In [ ]:
(df.loc[(df["INTERNATIONALFLAG"] == True) & (df["CONFIRMEDRISK"] == True)])["DESTINATIONCOUNTRY"].value_counts().head(10) #netherlands sem like a big thing, see what to make of this

In [ ]:
trans_us = df["TRANSACTIONONUS"].value_counts()

plt.bar(trans_us.index, trans_us.values)
plt.title("Transactions within and out of bank", y=1.03)
plt.show()

print(trans_us)

In [ ]:
trans_us = df.loc[df["CONFIRMEDRISK"] == True]["TRANSACTIONONUS"].value_counts()

plt.bar(trans_us.index, trans_us.values)
plt.title("Transactions within and out of bank (FRAUD ONLY)", y=1.03)
plt.show()

print(trans_us)

print(f'Share of transactions going out of bank {(len(df.loc[(df["CONFIRMEDRISK"] == True) & (df["TRANSACTIONONUS"] == False)]) / len(df.loc[df["CONFIRMEDRISK"] == True])) * 100:.2f} %')

In [ ]:
trans_us = df.loc[df["CONFIRMEDRISK"] == False]["TRANSACTIONONUS"].value_counts()

plt.bar(trans_us.index, trans_us.values)
plt.title("Transactions within and out of bank (NO FRAUD)", y=1.03)
plt.show()

print(trans_us)

print(f'Share of transactions going out of bank {(len(df.loc[(df["CONFIRMEDRISK"] == False) & (df["TRANSACTIONONUS"] == False)]) / len(df.loc[df["CONFIRMEDRISK"] == False])) * 100:.2f} %')

## Temporal EDA

In [ ]:
# Convert EVENTTIME to datetime if not already
df['EVENTTIME'] = pd.to_datetime(df['EVENTTIME'])
df['date'] = df['EVENTTIME'].dt.date

# Daily fraud rate
daily_fraud = df.groupby('date').agg({
    'CONFIRMEDRISK': ['count', 'sum']
}).round(4)
daily_fraud.columns = ['total_transactions', 'fraud_count']
daily_fraud['fraud_rate'] = daily_fraud['fraud_count'] / daily_fraud['total_transactions']


plt.figure(figsize=(15, 6))
plt.plot(daily_fraud.index, daily_fraud['fraud_rate'], linewidth=1)
plt.xlabel('Date')
plt.ylabel('Fraud Rate')
plt.title('Daily Fraud Rate Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Extract hour from timestamp
df['hour'] = df['EVENTTIME'].dt.hour

# Hourly fraud rate
hourly_fraud = df.groupby('hour').agg({
    'CONFIRMEDRISK': ['count', 'sum']
})
hourly_fraud.columns = ['total_transactions', 'fraud_count']
hourly_fraud['fraud_rate'] = hourly_fraud['fraud_count'] / hourly_fraud['total_transactions']

# Plot hourly pattern
plt.figure(figsize=(12, 6))
plt.plot(hourly_fraud.index, hourly_fraud['fraud_rate'], marker='o', linewidth=2)
plt.xlabel('Hour of Day')
plt.ylabel('Fraud Rate')
plt.title('Fraud Rate by Hour of Day')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)

# Highlight nighttime (e.g., 10pm - 6am)
plt.axvspan(22, 24, alpha=0.2, color='navy', label='Night (10pm-12am)')
plt.axvspan(0, 6, alpha=0.2, color='navy', label='Night (12am-6am)')
plt.legend()
plt.tight_layout()
plt.show() #higher in nighttime

In [ ]:
# Monthly fraud rate
df['year_month'] = df['EVENTTIME'].dt.to_period('M')

monthly_fraud = df.groupby('year_month').agg({
    'CONFIRMEDRISK': ['count', 'sum']
})
monthly_fraud.columns = ['total_transactions', 'fraud_count']
monthly_fraud['fraud_rate'] = monthly_fraud['fraud_count'] / monthly_fraud['total_transactions']

# Plot monthly trend
plt.figure(figsize=(15, 6))
plt.plot(monthly_fraud.index.astype(str), monthly_fraud['fraud_rate'], marker='o')
plt.xlabel('Month')
plt.ylabel('Fraud Rate')
plt.title('Monthly Fraud Rate Trend')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show() #spurious

In [ ]:
# Define nighttime hours
nighttime_hours = list(range(22, 24)) + list(range(0, 6))
df['is_nighttime'] = df['hour'].isin(nighttime_hours) #feature engineering here already

# Nighttime vs daytime fraud rate over time
night_day_fraud = df.groupby(['date', 'is_nighttime']).agg({
    'CONFIRMEDRISK': ['count', 'sum']
})
night_day_fraud.columns = ['total_transactions', 'fraud_count']
night_day_fraud['fraud_rate'] = night_day_fraud['fraud_count'] / night_day_fraud['total_transactions']
night_day_fraud = night_day_fraud.reset_index()

# Plot nighttime vs daytime trends
plt.figure(figsize=(15, 6))
night_data = night_day_fraud[night_day_fraud['is_nighttime'] == True]
day_data = night_day_fraud[night_day_fraud['is_nighttime'] == False]

plt.plot(night_data['date'], night_data['fraud_rate'], label='Nighttime (10pm-6am)', alpha=0.7)
plt.plot(day_data['date'], day_data['fraud_rate'], label='Daytime (6am-10pm)', alpha=0.7)

plt.xlabel('Date')
plt.ylabel('Fraud Rate')
plt.title('Fraud Rate: Nighttime vs Daytime Over Time')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Splitting the data



In [ ]:


TARGET_COL = "CONFIRMEDRISK"
TIME_COL = "EVENTTIME"


TRAIN_START = pd.Timestamp("2024-11-01")
VAL_START   = pd.Timestamp("2025-09-01")
TEST_START  = pd.Timestamp("2025-12-01")
TEST_END    = pd.Timestamp("2026-04-01")   
LABEL_MATURITY_CUTOFF = pd.Timestamp("2026-04-01")  


def temporal_split(df: pd.DataFrame):
    
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values(TIME_COL).reset_index(drop=True)

    
    df = df[df[TIME_COL] >= TRAIN_START]

    train = df[(df[TIME_COL] >= TRAIN_START) & (df[TIME_COL] < VAL_START)]
    val   = df[(df[TIME_COL] >= VAL_START)   & (df[TIME_COL] < TEST_START)]
    test  = df[(df[TIME_COL] >= TEST_START)  & (df[TIME_COL] < TEST_END)]

    return train, val, test


def xy(frame: pd.DataFrame):
    """Split a frame into (X, y). Drops target and the raw timestamp column
    from features — keep engineered time features (hour, weekday, etc.)
    instead of leaking the raw timestamp into the model."""
    X = frame.drop(columns=[TARGET_COL, TIME_COL])
    y = frame[TARGET_COL]
    return X, y


def summarise(name: str, frame: pd.DataFrame):
    if len(frame) == 0:
        print(f"{name:<10} | empty")
        return
    fraud_rate = frame[TARGET_COL].mean()
    print(
        f"{name:<10} | rows={len(frame):>9,} | "
        f"{frame[TIME_COL].min().date()} -> {frame[TIME_COL].max().date()} | "
        f"fraud rate={fraud_rate:.4%}"
    )

train, val, test = temporal_split(df)

print("Split sizes and fraud rates:")
summarise("train",    train)
summarise("val",      val)
summarise("test",     test)

X_train, y_train = xy(train)
X_val,   y_val   = xy(val)
X_test,  y_test  = xy(test)


In [ ]:
df["is_new_destinationcountry_for_customer"] = (
    df.groupby(["CUSTOMERID", "DESTINATIONCOUNTRY"])
).cumcount().eq(0).astype(int)

In [ ]:

TARGET_COL = "CONFIRMEDRISK"
TIME_COL   = "EVENTTIME"


eda_scratch_cols = ["date", "hour", "year_month", "is_nighttime"]
df.drop(columns=[c for c in eda_scratch_cols if c in df.columns],
        inplace=True)

customer_accounts = df[["CUSTOMERID", "ACCOUNTID"]].drop_duplicates()
df = df.merge(
    customer_accounts.rename(columns={
        "ACCOUNTID": "COUNTERPARTYID",
        "CUSTOMERID": "COUNTER_CUSTOMERID",
    }),
    on="COUNTERPARTYID",
    how="left",
)
internal_mask = (
    (df["TRANSACTIONONUS"] == True)
    & (df["COUNTER_CUSTOMERID"] == df["CUSTOMERID"])
)
df = df[~internal_mask].drop(columns=["COUNTER_CUSTOMERID"])
print(f"Rows after removing own-account transfers: {len(df):,}")

TRAIN_START = pd.Timestamp("2024-11-01")
VAL_START   = pd.Timestamp("2025-09-01")
TEST_START  = pd.Timestamp("2025-12-01")
TEST_END    = pd.Timestamp("2026-04-01")

def temporal_split(df: pd.DataFrame):
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL])
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    df = df[df[TIME_COL] >= TRAIN_START]

    train = df[(df[TIME_COL] >= TRAIN_START) & (df[TIME_COL] < VAL_START)]
    val   = df[(df[TIME_COL] >= VAL_START)   & (df[TIME_COL] < TEST_START)]
    test  = df[(df[TIME_COL] >= TEST_START)  & (df[TIME_COL] < TEST_END)]
    return train, val, test


train_raw, val_raw, test_raw = temporal_split(df)

for name, split in [("Train", train_raw), ("Val", val_raw), ("Test", test_raw)]:
    fr = split[TARGET_COL].mean()
    print(
        f"{name:<6} | {len(split):>9,} rows | "
        f"{split[TIME_COL].min().date()} → {split[TIME_COL].max().date()} | "
        f"fraud rate {fr:.4%}"
    )

def add_amount_features(df):
    """Log-transform BASEVALUE and flag foreign currency."""
    df = df.copy()
    df["log_basevalue"]       = np.log1p(df["BASEVALUE"])
    df["is_foreign_currency"] = (df["CURRENCY"] != "EUR").astype(int)
    return df


def add_temporal_features(df):
    """Cyclical hour/day encoding + coarse boolean flags."""
    df = df.copy()
    hour = df[TIME_COL].dt.hour
    dow  = df[TIME_COL].dt.dayofweek

    df["hour_sin"]     = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]     = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"]      = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"]      = np.cos(2 * np.pi * dow / 7)
    df["is_nighttime"] = ((hour >= 22) | (hour < 6)).astype(int)
    df["is_weekend"]   = (dow >= 5).astype(int)
    return df


def add_velocity_features(df):
    """Time gaps between consecutive transactions per customer."""
    df = df.sort_values(["CUSTOMERID", TIME_COL]).copy()

    df["time_since_prev_txn"] = (
        df.groupby("CUSTOMERID")[TIME_COL].diff().dt.total_seconds()
    )
    df["log_time_since_prev_txn"] = np.log1p(df["time_since_prev_txn"])
    df["is_first_txn_for_customer"] = df["time_since_prev_txn"].isna().astype(int)

    df["customer_first_seen"] = (
        df.groupby("CUSTOMERID")[TIME_COL].transform("min")
    )
    df["time_since_first_txn"] = (
        (df[TIME_COL] - df["customer_first_seen"]).dt.total_seconds()
    )
    df.drop(columns=["customer_first_seen"], inplace=True)
    return df


def add_is_new_features(df):
    df = df.sort_values(["CUSTOMERID", TIME_COL]).copy()

    for field in ["DESTINATIONCOUNTRY", "COUNTERPARTYID", "CHANNEL", "PAYMENTSUBMETHOD"]:
        short = field.lower().replace("id", "")[:12]
        df[f"is_new_{short}_for_customer"] = (
            df.groupby(["CUSTOMERID", field]).cumcount().eq(0).astype(int)
        )

    df["is_new_intl_dest_for_customer"] = (
        df["is_new_destinationcountry_for_customer"].astype(bool)
        & df["INTERNATIONALFLAG"].astype(bool)
    ).astype(int)
    return df


def add_customer_aggregations(df, windows=("1D", "7D", "30D")):
    """Rolling count / mean / std / max / sum of BASEVALUE per customer."""
    df = df.sort_values(["CUSTOMERID", TIME_COL]).copy()
    df_idx = df.set_index(TIME_COL)

    result = {}
    for w in windows:
        g = df_idx.groupby("CUSTOMERID")["BASEVALUE"].rolling(w, closed="left")
        result[f"cust_amt_count_{w}"] = g.count().values
        result[f"cust_amt_mean_{w}"]  = g.mean().values
        result[f"cust_amt_std_{w}"]   = g.std().values
        result[f"cust_amt_max_{w}"]   = g.max().values
        result[f"cust_amt_sum_{w}"]   = g.sum().values

    for col, vals in result.items():
        df[col] = vals
    return df


def add_conditional_aggregations(df, group_col, windows=("1D", "7D", "30D")):
    """Rolling aggregations per (CUSTOMERID, group_col)."""
    df = df.sort_values(["CUSTOMERID", group_col, TIME_COL]).copy()
    df_idx = df.set_index(TIME_COL)
    short = group_col.lower().replace("id", "")[:10]

    result = {}
    for w in windows:
        g = df_idx.groupby(["CUSTOMERID", group_col])["BASEVALUE"].rolling(w, closed="left")
        result[f"cust_{short}_count_{w}"] = g.count().values
        result[f"cust_{short}_mean_{w}"]  = g.mean().values
        result[f"cust_{short}_sum_{w}"]   = g.sum().values

    for col, vals in result.items():
        df[col] = vals
    return df


def add_nighttime_share(df, windows=("7D", "30D")):
    df = df.sort_values(["CUSTOMERID", TIME_COL]).copy()

    if "is_nighttime" not in df.columns:
        hour = df[TIME_COL].dt.hour
        df["is_nighttime"] = ((hour >= 22) | (hour < 6)).astype(int)

    df_idx = df.set_index(TIME_COL)
    for w in windows:
        g = df_idx.groupby("CUSTOMERID")["is_nighttime"].rolling(w, closed="left")
        df[f"cust_night_share_{w}"] = g.mean().values
        df[f"cust_night_count_{w}"] = g.sum().values
    return df


def add_safe_pair_flag(df, train_end_ts):
    df = df.copy()
    history = df[df[TIME_COL] < train_end_ts]

    pair_stats = (
        history.groupby(["ACCOUNTID", "COUNTERPARTYID"])[TIME_COL]
        .agg(["min", "max", "count"])
    )
    pair_stats["duration"] = pair_stats["max"] - pair_stats["min"]
    safe = pair_stats[
        (pair_stats["duration"] > timedelta(days=14))
        & (pair_stats["count"] >= 2)
    ].reset_index()[["ACCOUNTID", "COUNTERPARTYID"]]
    safe["IS_SAFE_PAIR"] = 1

    df = df.merge(safe, on=["ACCOUNTID", "COUNTERPARTYID"], how="left")
    df["IS_SAFE_PAIR"] = df["IS_SAFE_PAIR"].fillna(0).astype(int)
    return df


def add_transaction_to_ratio(df, train_mean, train_median):
    df = df.copy()
    df["txn_to_avg_ratio"]    = df["BASEVALUE"] / train_mean
    df["txn_to_median_ratio"] = df["BASEVALUE"] / train_median
    return df

def build_rolling_features(train, val, test):

    train = train.copy(); train["_split"] = "train"
    val   = val.copy();   val["_split"]   = "val"
    test  = test.copy();  test["_split"]  = "test"
    
    combined = pd.concat([train, val, test], ignore_index=True)
    combined = combined.sort_values([TIME_COL]).reset_index(drop=True)

    combined = add_amount_features(combined)
    combined = add_temporal_features(combined)

    combined = add_velocity_features(combined)
    combined = add_is_new_features(combined)
    combined = add_customer_aggregations(combined)
    combined = add_conditional_aggregations(combined, "DESTINATIONCOUNTRY")
    combined = add_conditional_aggregations(combined, "COUNTERPARTYID")
    combined = add_conditional_aggregations(combined, "PAYMENTSUBMETHOD")
    combined = add_nighttime_share(combined)

    train_out = combined[combined["_split"] == "train"].drop(columns="_split")
    val_out   = combined[combined["_split"] == "val"].drop(columns="_split")
    test_out  = combined[combined["_split"] == "test"].drop(columns="_split")
    return train_out, val_out, test_out


train_feat, val_feat, test_feat = build_rolling_features(
    train_raw, val_raw, test_raw
)
print(f"Features after rolling: {train_feat.shape[1]} columns")

train_mean   = train_feat["BASEVALUE"].mean()
train_median = train_feat["BASEVALUE"].median()

train_feat = add_transaction_to_ratio(train_feat, train_mean, train_median)
val_feat   = add_transaction_to_ratio(val_feat,   train_mean, train_median)
test_feat  = add_transaction_to_ratio(test_feat,  train_mean, train_median)


train_feat = add_safe_pair_flag(train_feat, VAL_START)
val_feat   = add_safe_pair_flag(val_feat,   VAL_START)
test_feat  = add_safe_pair_flag(test_feat,  VAL_START)

class FraudEncoder:
    RARE_GROUPS = {
        "ACCOUNTBRANCHID":   20,
        "COUNTERAGENTID":    20,
        "CURRENCY":           5,
        "DESTINATIONCOUNTRY": 5,
    }

    TARGET_ENC_COLS = [
        "ACCOUNTBRANCHID",
        "COUNTERAGENTID",
        "COUNTERPARTYID",
        "ACCAGENTCOUNTRY",
    ]

    OHE_COLS = [
        "CHANNEL",
        "PAYMENTCLEARING",
        "PAYMENTSUBMETHOD",
        "CURRENCY",
        "DESTINATIONCOUNTRY",
        "COUNTERIDFORMAT",
    ]

    BOOL_COLS = ["INTERNATIONALFLAG", "TRANSACTIONONUS"]

    def __init__(self):
        self.top_values_ = {}
        self.target_encoders_ = {}
        self.ohe_ = None
        self.ohe_cols_present_ = []


    def fit(self, df_train):
        for col, n in self.RARE_GROUPS.items():
            if col in df_train.columns:
                self.top_values_[col] = set(
                    df_train[col].value_counts().head(n).index
                )

        df_tmp = self._apply_rare_grouping(df_train.copy())
        for col in self.TARGET_ENC_COLS:
            if col in df_tmp.columns:
                enc = TargetEncoder(
                    smooth="auto",
                    cv=5,
                    random_state=42,
                )
                encoded = enc.fit_transform(
                    df_tmp[[col]], df_tmp[TARGET_COL]
                )
                self.target_encoders_[col] = {
                    "encoder": enc,
                    "train_encoded": encoded.ravel(),
                }

        self.ohe_cols_present_ = [
            c for c in self.OHE_COLS if c in df_tmp.columns
        ]
        if self.ohe_cols_present_:
            self.ohe_ = OneHotEncoder(
                handle_unknown="ignore", sparse_output=False, drop="first"
            )
            self.ohe_.fit(df_tmp[self.ohe_cols_present_])
        return self

    def transform(self, df, split="test"):
        df = self._apply_rare_grouping(df.copy())
        df = self._cast_bools(df)

        for col, bundle in self.target_encoders_.items():
            if col not in df.columns:
                continue
            if split == "train":
                df[f"{col.lower()}_te"] = bundle["train_encoded"]
            else:
                df[f"{col.lower()}_te"] = (
                    bundle["encoder"].transform(df[[col]]).ravel()
                )

        if self.ohe_ is not None and self.ohe_cols_present_:
            ohe_arr = self.ohe_.transform(df[self.ohe_cols_present_])
            ohe_names = self.ohe_.get_feature_names_out(self.ohe_cols_present_)
            ohe_df = pd.DataFrame(ohe_arr, index=df.index, columns=ohe_names)
            df = pd.concat([df, ohe_df], axis=1)

        cols_to_drop = (
            list(self.RARE_GROUPS.keys())
            + self.TARGET_ENC_COLS
            + self.ohe_cols_present_
        )
        cols_to_drop = list(set(c for c in cols_to_drop if c in df.columns))
        df.drop(columns=cols_to_drop, inplace=True)

        return df

    def _apply_rare_grouping(self, df):
        for col, tops in self.top_values_.items():
            if col in df.columns:
                df[col] = df[col].where(df[col].isin(tops), other="RARE")
        return df

    def _cast_bools(self, df):
        for col in self.BOOL_COLS:
            if col in df.columns:
                if df[col].dtype == object:
                    df[col] = (
                        df[col].astype(str).str.strip().str.lower()
                        .map({"true": 1, "false": 0})
                    )
                else:
                    df[col] = df[col].astype(int)
        return df


encoder = FraudEncoder()
encoder.fit(train_feat)

train_enc = encoder.transform(train_feat, split="train")
val_enc   = encoder.transform(val_feat,   split="val")
test_enc  = encoder.transform(test_feat,  split="test")

DROP_FROM_X = [
    TARGET_COL,
    TIME_COL,
    "TRANSACTIONID",  
    "ACCOUNTID",      
    "CUSTOMERID",     
    "COUNTERPARTYID", 
]


def xy(frame):
    drop = [c for c in DROP_FROM_X if c in frame.columns]
    X = frame.drop(columns=drop)
    y = frame[TARGET_COL].astype(int)
    return X, y


X_train, y_train = xy(train_enc)
X_val,   y_val   = xy(val_enc)
X_test,  y_test  = xy(test_enc)

print(f"\nFinal shapes:")
print(f"  X_train: {X_train.shape}   fraud rate: {y_train.mean():.4%}")
print(f"  X_val:   {X_val.shape}   fraud rate: {y_val.mean():.4%}")
print(f"  X_test:  {X_test.shape}   fraud rate: {y_test.mean():.4%}")


obj_cols = X_train.select_dtypes("object").columns.tolist()
if obj_cols:
    print(f"\n Object columns still present — need encoding: {obj_cols}")
else:
    print("\n  All features are numeric.")

